### CountryStatistics 클래스

* clean_numeric_data(value: str): 문자열을 입력으로 받아 쉼표, 달러 기호, 백분율 기호를 제거하고 숫자 값을 float으로 반환합니다. 입력을 변환할 수 없으면 0.0을 반환합니다.
힌트: 이 함수는 쉼표나 달러 기호가 포함된 CSV 파일의 숫자 데이터를 처리하는 데 유용합니다. 빈 문자열이나 숫자가 아닌 값을 처리할 수 있도록 예외 처리에 신경 쓰세요.

* load_data(filename: str): CSV 파일에서 국가 데이터를 로드하고 데이터를 파싱하여 Country 객체를 생성합니다. 정리된 데이터는 self.countries 리스트에 저장됩니다. 파일 로드 완료 후 현재 시간을 self.load_time에 저장합니다.
힌트: csv.reader를 사용하여 CSV 파일을 한 줄씩 읽습니다. 첫 번째 행은 헤더이므로 건너뜁니다. 모든 숫자 데이터 필드에 대해 clean_numeric_data를 호출하여 데이터를 저장하기 전에 정확하게 정리된 데이터를 확인하세요. GDP, 기대 수명, 인구 등의 열 인덱스를 정확하게 매핑하세요.

* top_5_gdp(): GDP가 가장 높은 상위 5개 국가의 이름과 쉼표로 형식화된 GDP 값을 포함한 튜플 목록을 반환합니다.
힌트: 국가들을 gdp 속성 기준으로 내림차순 정렬하세요. 슬라이싱 ([:5])을 사용하여 상위 5개 항목을 선택하세요. 국가 이름과 형식화된 GDP 값을 반환하세요.

* top_5_life_expectancy(): 기대 수명이 가장 긴 상위 5개 국가의 이름과 그들의 기대 수명 값을 포함한 튜플 목록을 반환합니다.
힌트: 국가들을 기대 수명(life_expectancy) 기준으로 내림차순 정렬하세요. 슬라이싱 ([:5])을 사용하여 상위 5개 항목을 추출하세요. 기대 수명 값은 형식화할 필요가 없으므로 있는 그대로 반환하세요.

* top_5_density(): 인구 밀도가 가장 높은 상위 5개 국가의 이름과 인구 밀도를 반환합니다.
힌트: 국가들을 인구 밀도(density) 속성 기준으로 내림차순 정렬하세요. 슬라이싱 ([:5])을 사용하여 상위 5개 항목을 선택하세요. 인구 밀도 값은 정수이므로 형식화할 필요가 없습니다.

* get_data_load_time(format_str=None): 데이터 로드 시간을 반환하며, format_str로 사용자 지정 포맷을 지원합니다. 데이터가 아직 로드되지 않았다면 None을 반환합니다.
힌트: strftime 메서드를 이용해 봅니다.

* plot_geo_scatter(self): 위도와 경도 데이터를 사용하여 지리적 산점도를 그립니다.
힌트: plotly.express.scatter_geo 메서드를 이용해 봅니다. (16.2 이후 도전하기)

In [ ]:
import csv
from dataclasses import dataclass
import pandas as pd
import plotly.express as px
from datetime import datetime

@dataclass
class Country:
    name: str
    gdp: float
    life_expectancy: float
    density: float
    lat: float
    lon: float

class CountryStatistics:
    def __init__(self, filename:str):
        self.filename = filename
        self.countries = []
        self.load_time = load_time = None
        
    def clean_numeric_data(self, value: str) -> float:
        try:
            clean = value.replace(",", "").replace("$", "").replace("%", "")
            return float(clean)
        except (ValueError, AttributeError):
            return 0.0
    
    def load_data(self):
        with open(self.filename, encoding='utf-8') as f:
            reader = csv.DictReader(f)  # 딕셔너리 형태로 읽기

            for row in reader:
                try:
                    name = row["Country"]
                    pop = self.clean_numeric_data(row["Population"])
                    area = self.clean_numeric_data(row["Land Area(Km2)"])
                    lat = self.clean_numeric_data(row["Latitude"])
                    lon = self.clean_numeric_data(row["Longitude"])

                    density = pop / area if area > 0 else 0

                    country = Country(
                        name=name,
                        gdp=0.0,  # GDP 정보 없음
                        life_expectancy=0.0,  # 기대 수명 없음
                        density=density,
                        lat=lat,
                        lon=lon
                    )
                    self.countries.append(country)

                except (KeyError, ValueError):
                    continue

        self.load_time = datetime.now()
            
            

                    
    
    def top_5_gdp(self):
        sorted_countries = sorted(self.countries, key=lambda c: c.gdp, reverse=True)
        return [(c.name, f"{c.gdp:,.2f}") for c in sorted_countries[:5]]

    def top_5_life_expectancy(self):
        sorted_countries = sorted(self.countries, key=lambda c: c.life_expectancy, reverse=True)
        return [(c.name, c.life_expectancy) for c in sorted_countries[:5]]

    def top_5_density(self):
        sorted_countries = sorted(self.countries, key=lambda c: c.density, reverse=True)
        return [(c.name, int(c.density)) for c in sorted_countries[:5]]

    def get_data_load_time(self, format_str=None):
        if not self.load_time:
            return None
        return self.load_time.strftime(format_str) if format_str else str(self.load_time)

    def plot_geo_scatter(self):
        df = pd.DataFrame([{
            'Country': c.name,
            'Latitude': c.lat,
            'Longitude': c.lon,
            'GDP': c.gdp
        } for c in self.countries])

        fig = px.scatter_geo(df,
                            at='Latitude',
                            lon='Longitude',
                            hover_name='Country',
                            size='GDP',
                            projection='natural earth',
                            title='Global GDP by Country')
        fig.show()

stats = CountryStatistics("world-data-2023.csv")
stats.load_data()

print("💰 Top 5 GDP Countries:")
print(stats.top_5_gdp())

print("\n👵 Top 5 Life Expectancy Countries:")
print(stats.top_5_life_expectancy())

print("\n🏙️ Top 5 Density Countries:")
print(stats.top_5_density())

print("\n⏰ Data Load Time:")
print(stats.get_data_load_time("%Y-%m-%d %H:%M:%S"))

# 🌍 지리적 시각화 (옵션)
# stats.plot_geo_scatter()

💰 Top 5 GDP Countries:
[('Afghanistan', '0.00'), ('Albania', '0.00'), ('Algeria', '0.00'), ('Andorra', '0.00'), ('Angola', '0.00')]

👵 Top 5 Life Expectancy Countries:
[('Afghanistan', 0.0), ('Albania', 0.0), ('Algeria', 0.0), ('Andorra', 0.0), ('Angola', 0.0)]

🏙️ Top 5 Density Countries:
[('Monaco', 19482), ('Singapore', 7965), ('Bahrain', 1962), ('Maldives', 1781), ('Malta', 1590)]

⏰ Data Load Time:
2025-05-07 10:06:54
